<a href="https://colab.research.google.com/github/sammik9660/FoodSpoilageMonitor/blob/master/mltest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# [CELL 01] Google Apps Script 백엔드 전체 테스트
# ============================================================

import requests
import pandas as pd
from io import StringIO

WEB_APP_URL = (
    "https://script.google.com/macros/s/"
    "AKfycbyrDRFYcw6KKLK2fae04ZW1nxsuBVlW9LuAaUawLq3Cxim1Qri7uZe5AEzsPwf3txLd/exec"
)

# ------------------------------------------------------------
# ESP32 대신 가짜 센서 데이터 전송
# ------------------------------------------------------------

test_payload = {
    "temperature": 24.50,
    "humidity": 55.20,
    "pressure": 1012.30,
    "gas": 130.10
}

print("=== TEST POST ===")

r = requests.post(
    WEB_APP_URL,
    json=test_payload,
    timeout=30
)

print("HTTP:", r.status_code)
print("Final URL:", r.url)
print("Response:", r.text[:500])

if '"success":true' in r.text.replace(" ", "").lower():
    print("\n✅ Apps Script POST 성공")
else:
    print("\n⚠️ success:true가 확인되지 않음")


# ------------------------------------------------------------
# CSV 다시 받아오기
# ------------------------------------------------------------

print("\n=== CSV DOWNLOAD TEST ===")

csv_r = requests.get(
    WEB_APP_URL,
    params={"download": "csv"},
    timeout=30
)

print("HTTP:", csv_r.status_code)
print("Content-Type:", csv_r.headers.get("content-type"))

csv_text = csv_r.text.lstrip("\ufeff")

print("\n--- CSV 앞부분 ---")
print(csv_text[:500])

# HTML이 돌아왔으면 CSV 엔드포인트 문제
if "<html" in csv_text[:300].lower():
    raise RuntimeError(
        "CSV 대신 HTML 페이지가 반환되었습니다. "
        "Apps Script의 doGet(download=csv) 부분을 확인해야 합니다."
    )

df = pd.read_csv(StringIO(csv_text))

print("\n✅ CSV → pandas 로딩 성공")
display(df.tail())

In [ ]:
# ============================================================
# [CELL 02] 데이터 구조 / 품질 검사
# ============================================================

import numpy as np
import pandas as pd

if "gas" in df.columns and "gas_resistance" not in df.columns:
    df = df.rename(columns={"gas": "gas_resistance"})

required_columns = [
    "timestamp",
    "temperature",
    "humidity",
    "pressure",
    "gas_resistance"
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(f"필수 컬럼 없음: {missing}")

# Apps Script 날짜 문자열 끝의
# "(Korean Standard Time)" 같은 설명 제거
df["timestamp"] = (
    df["timestamp"]
    .astype(str)
    .str.replace(r"\s*\([^)]*\)\s*$", "", regex=True)
)

# UTC 기준 datetime으로 변환
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce",
    utc=True
)

numeric_columns = [
    "temperature",
    "humidity",
    "pressure",
    "gas_resistance"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.sort_values("timestamp").reset_index(drop=True)

print("행 개수:", len(df))

print("\n[결측값]")
print(df[required_columns].isna().sum())

print("\n[기본 통계]")
display(df[numeric_columns].describe())

print("\n[최근 데이터]")
display(df.tail())

In [ ]:
# ============================================================
# [CELL 03] 시계열 그래프
# ============================================================

import matplotlib.pyplot as plt

plot_columns = {
    "temperature": "Temperature (°C)",
    "humidity": "Humidity (%RH)",
    "pressure": "Pressure (hPa)",
    "gas_resistance": "Gas Resistance"
}

for column, ylabel in plot_columns.items():

    plt.figure(figsize=(12, 4))

    plt.plot(
        df["timestamp"],
        df[column]
    )

    plt.xlabel("Time")
    plt.ylabel(ylabel)
    plt.title(ylabel)

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# [CELL 04] 정제 데이터 저장
# ============================================================

df.to_csv(
    "sensor_data_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ sensor_data_clean.csv 저장 완료")